**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Triple-Regime (A/B/C) — Noise-Free Training

Identical architecture to `TripleRegime_ABC_Train.ipynb` but trained on
the **noise-free dictionary** (`QuasiRand_t2_200.mat`) only.

| | Mixed-SNR (original) | Noise-free (this notebook) |
|--|--|--|
| Training data | 4 × noisy SNR levels | Noise-free dictionary only |
| Input dim | 43 | 43 |
| Architecture | TripleRegimeModel | TripleRegimeModel (identical) |
| Output dir | `triple_regime_results_v1` | `triple_regime_nf_results_v1` |

**Input layout**: `[L2-norm signal (40)] + [feat_A] + [feat_B] + [feat_C]` → 43 dims  
**Output**: SO₂, CBV, R, T2 in [0,1]

---

## 1. Imports & GPU

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DM       = '#E65100'
C_DL_NF    = '#2E7D32'
C_DL_NOISY = '#1565C0'
C_TRIPLE   = '#6A1B9A'
C_TRIPLE_NF = '#AD1457'   # deep pink — distinguishes NF variant

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration

In [ ]:
CONFIG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    'param_path'        : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'echotimes_path'    : '../echotimes.mat',
    'output_dir'        : './results/triple_regime_nf_results_v1',
    'dict_key'          : 'Dico40_save',
    'param_key'         : 'par_save',

    # ── Parameter space ────────────────────────────────────────────────────────
    'param_mins'  : np.array([0.0,    0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,    0.15,   25.0e-6,  0.200]),
    'param_names' : ['SO2', 'CBV', 'R', 'T2'],

    # ── GESFIDE geometry ───────────────────────────────────────────────────────
    'n_fid'   : 14,   # Part A: echoes  0–13
    'n_rephas': 16,   # Part B: echoes 14–29  (SE = echo 29)
    'n_postse': 10,   # Part C: echoes 30–39

    # ── Feature scaling (must match the noisy model for fair comparison) ───────
    'R2starA_min':  2.0,   'R2starA_max': 55.0,
    'R2starB_min': -30.0,  'R2starB_max': 22.0,
    'R2starC_min':  2.0,   'R2starC_max': 55.0,

    # ── Training ───────────────────────────────────────────────────────────────
    # No SNR injection — full noise-free dictionary used as-is
    'test_frac'    : 0.15,
    'val_frac'     : 0.15,
    'batch_size'   : 16384,
    'epochs'       : 200,
    'patience'     : 25,
    'lr'           : 5e-5,
    'dropout'      : 0.05,
    'param_weights': [1.0, 12.0, 8.0, 1.0],

    # ── Evaluation SNR levels (noisy test sets used for RMSE vs SNR curves) ───
    'dict_base_path': '../subsamples/subsamples_v3',
    'snr_levels'    : [20, 50, 100, 150],
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']   # = 30
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

PARAM_NAMES = ['SO₂', 'CBV', 'R',   'T2']
PARAM_UNITS = ['(%)', '(%)', '(µm)', '(ms)']
PARAM_SCALE = [100,   100,   1e6,    1000]
PKEYS       = CONFIG['param_names']

print(f'SE_ECHO = {SE_ECHO}')
print(f'Input dim: 40 + 3 = 43  (L2-norm signal + feat_A + feat_B + feat_C)')
print(f'Output: {CONFIG["output_dir"]}')

## 3. Load echo times

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A     = echo_times_s[:CONFIG['n_fid']]
T_B     = echo_times_s[CONFIG['n_fid']:SE_ECHO]
T_C     = echo_times_s[SE_ECHO:]
T_SE_S  = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S

print(f'Part A: echoes  0–{CONFIG["n_fid"]-1},  t=[{T_A[0]*1e3:.2f},...,{T_A[-1]*1e3:.2f}] ms')
print(f'Part B: echoes {CONFIG["n_fid"]}–{SE_ECHO-1},  t=[{T_B[0]*1e3:.2f},...,{T_B[-1]*1e3:.2f}] ms')
print(f'Part C: echoes {SE_ECHO}–{SE_ECHO+CONFIG["n_postse"]-1},  t_rel=[{T_C_rel[0]*1e3:.2f},...,{T_C_rel[-1]*1e3:.2f}] ms')
print(f'Spin echo at {T_SE_S*1e3:.2f} ms')

## 4. Utilities

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    if (~mask).sum():
        print(f'  Filtered {(~mask).sum()} out-of-range ({(~mask).mean()*100:.1f}%)')
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    if (~valid).sum(): print(f'  Removed {(~valid).sum()} non-finite entries')
    return signals[valid], params[valid]

def batched_predict(model, x_np, batch_size=4096, dev=device):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext == 'png' else None)
    print(f'  Saved: {name}')

print('Utilities ready.')

## 5. Triple-regime feature functions

Identical to `TripleRegime_ABC_Train.ipynb` — shared across noisy and noise-free models.

In [ ]:
def ols_slope(t_vec, sig_mat):
    log_s  = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t      = t_vec.astype(np.float64); t_c = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()


def compute_triple_regime_features(sig_raw, config):
    """
    R2*_A = R2 + R2'  (Part A, always positive)
    R2*_B = R2 - R2'  (Part B, raw signed)
    R2*_C = R2 + R2'  (Part C, relative to SE, always positive)
    """
    n_fid   = config['n_fid']
    se_echo = n_fid + config['n_rephas']
    r2_lb   = 1.0 / config['param_maxs'][3]

    slope_A = ols_slope(T_A,     sig_raw[:, :n_fid])
    slope_B = ols_slope(T_B,     sig_raw[:, n_fid:se_echo])
    slope_C = ols_slope(T_C_rel, sig_raw[:, se_echo:])

    R2starA = np.maximum(-slope_A, r2_lb).astype(np.float32)
    R2starB = (-slope_B).astype(np.float32)              # raw signed — no clip
    R2starC = np.maximum(-slope_C, r2_lb).astype(np.float32)

    return R2starA, R2starB, R2starC


def scale_triple_features(R2starA, R2starB, R2starC, config):
    def sc(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    return (sc(R2starA, config['R2starA_min'], config['R2starA_max']),
            sc(R2starB, config['R2starB_min'], config['R2starB_max']),
            sc(R2starC, config['R2starC_min'], config['R2starC_max']))


def build_43dim_input(sig_raw, config):
    """Full pipeline: raw signal → 43-dim model input."""
    R2A, R2B, R2C = compute_triple_regime_features(sig_raw, config)
    fA, fB, fC    = scale_triple_features(R2A, R2B, R2C, config)
    sig_norm       = euclidean_norm(sig_raw)
    return np.concatenate([sig_norm, fA[:, None], fB[:, None], fC[:, None]], axis=1)


print('Triple-regime feature functions ready.')

## 6. Dataset preparation — noise-free only

Key difference from the noisy notebook: **no SNR loop**.
The noise-free dictionary is loaded once and used directly.

In [ ]:
print('Loading noise-free signals...')
sig_raw  = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
par_raw  = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]

# Filter and clean
sig_raw, par_raw = filter_param_range(
    sig_raw, par_raw, CONFIG['param_mins'], CONFIG['param_maxs'])
sig_raw, par_raw = clean_data(sig_raw, par_raw)
print(f'Loaded {len(sig_raw):,} noise-free samples')

# Feature sanity check on a small subset
rng = np.random.default_rng(42)
idx_check = rng.choice(len(sig_raw), 20_000, replace=False)
R2A_chk, R2B_chk, R2C_chk = compute_triple_regime_features(sig_raw[idx_check], CONFIG)
fA_chk, fB_chk, fC_chk    = scale_triple_features(R2A_chk, R2B_chk, R2C_chk, CONFIG)
print(f'Feature check (noise-free, N=20k):')
print(f'  R2*_A: [{R2A_chk.min():.1f}, {R2A_chk.max():.1f}] s^-1  feat_A: [{fA_chk.min():.3f}, {fA_chk.max():.3f}]')
print(f'  R2*_B: [{R2B_chk.min():.1f}, {R2B_chk.max():.1f}] s^-1  feat_B: [{fB_chk.min():.3f}, {fB_chk.max():.3f}]  (neg={( R2B_chk<0).mean()*100:.1f}%)')
print(f'  R2*_C: [{R2C_chk.min():.1f}, {R2C_chk.max():.1f}] s^-1  feat_C: [{fC_chk.min():.3f}, {fC_chk.max():.3f}]')

# Build full 43-dim input matrix
print('\nBuilding 43-dim input...')
X = build_43dim_input(sig_raw, CONFIG).astype(np.float32)   # (N, 43)
Y = params_scale(par_raw[:, :4],
                  CONFIG['param_mins'][:4],
                  CONFIG['param_maxs'][:4]).astype(np.float32)

# Clean after feature computation (features may introduce NaN at boundaries)
valid = np.all(np.isfinite(X), axis=1) & np.all(np.isfinite(Y), axis=1)
X, Y  = X[valid], Y[valid]
print(f'Dataset: {len(X):,} samples  input_dim={X.shape[1]}  output_dim={Y.shape[1]}')

# Train / val / test split
x_tv, x_test, y_tv, y_test = train_test_split(
    X, Y, test_size=CONFIG['test_frac'], random_state=42)
x_train, x_val, y_train, y_val = train_test_split(
    x_tv, y_tv, test_size=CONFIG['val_frac'], random_state=42)

y_test_raw = params_inverse(y_test, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

print(f'Train={len(x_train):,}  Val={len(x_val):,}  Test={len(x_test):,}')
print(f'feat_A range: [{x_train[:,40].min():.3f}, {x_train[:,40].max():.3f}]')
print(f'feat_B range: [{x_train[:,41].min():.3f}, {x_train[:,41].max():.3f}]')
print(f'feat_C range: [{x_train[:,42].min():.3f}, {x_train[:,42].max():.3f}]')

## 7. Model architecture — identical to noisy notebook

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim)
        b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        p = self.net(cond)
        return p[:, :self.feature_dim] * x + p[:, self.feature_dim:]


class TripleRegimeModel(nn.Module):
    """
    Conv1D backbone + FiLM conditioning on (R2*_A, R2*_B, R2*_C).
    Input:  (B, 43)  Output: (B, 4)
    """
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128,3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        ch = film_cond_hidden
        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, 3, ch)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, 3, ch)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, 3, ch)
        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)
        cond = x[:, 40:43]
        c = self.conv(echo).flatten(1)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


_m = TripleRegimeModel().to(device)
_x = torch.randn(8, 43).to(device)
print(f'Output shape: {_m(_x).shape}   (expected [8, 4])')
print(f'Parameters:   {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x

## 8. Loss & training loop

In [ ]:
class WeightedMAELoss(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.register_buffer('w', torch.tensor(weights, dtype=torch.float32))
    def forward(self, pred, target):
        return (torch.abs(pred - target) * self.w.unsqueeze(0)).sum(dim=1).mean()


def train_model(model, x_train, y_train, x_val, y_val,
                config, ckpt_name='triple_nf_best.pt'):
    criterion = WeightedMAELoss(config['param_weights']).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=8, factor=0.5, min_lr=1e-7, verbose=True)

    x_t = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    x_v = torch.tensor(x_val,   dtype=torch.float32).to(device)
    y_v = torch.tensor(y_val,   dtype=torch.float32).to(device)

    loader   = DataLoader(TensorDataset(x_t, y_t),
                           batch_size=config['batch_size'], shuffle=True)
    best_val = float('inf'); patience_cnt = 0
    history  = {'train_loss': [], 'val_loss': []}
    ckpt_path = os.path.join(config['output_dir'], 'models', ckpt_name)

    print(f'Training up to {config["epochs"]} epochs  (patience={config["patience"]})')
    print(f'Batch size: {config["batch_size"]:,}  Batches/epoch: {len(loader):,}')

    for epoch in range(1, config['epochs'] + 1):
        model.train(); t0 = time.time(); tloss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item() * len(xb)
        tloss /= len(x_t)

        model.eval()
        with torch.no_grad():
            vloss = criterion(model(x_v), y_v).item()

        scheduler.step(vloss)
        history['train_loss'].append(tloss)
        history['val_loss'].append(vloss)

        if vloss < best_val:
            best_val = vloss; patience_cnt = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_cnt += 1

        if epoch % 10 == 0 or epoch <= 5:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch:4d}  train={tloss:.5f}  val={vloss:.5f}  '
                  f'best={best_val:.5f}  lr={lr:.2e}  ({time.time()-t0:.1f}s)')

        if patience_cnt >= config['patience']:
            print(f'Early stop at epoch {epoch}')
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'\nBest val={best_val:.6f}  loaded: {ckpt_path}')
    return model, history


print('Loss and training loop ready.')

## 9. Train

In [ ]:
model = TripleRegimeModel(
    n_outputs=4,
    dropout=CONFIG['dropout']
).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

model, history = train_model(
    model, x_train, y_train, x_val, y_val, CONFIG)

## 10. Training curve

In [ ]:
fig_tc, ax_tc = plt.subplots(figsize=(7, 3.5))
ep = len(history['train_loss'])
ax_tc.plot(range(1, ep+1), history['train_loss'], color=C_TRIPLE_NF, lw=1.5, label='Train')
ax_tc.plot(range(1, ep+1), history['val_loss'],   color=C_DL_NF,    lw=1.5, label='Val', ls='--')
ax_tc.set_xlabel('Epoch', fontsize=10)
ax_tc.set_ylabel('Weighted MAE', fontsize=10)
ax_tc.set_title('Triple-Regime NF (A+B+C) — Training Curve', fontsize=11, fontweight='bold')
ax_tc.legend()
ax_tc.spines['top'].set_visible(False); ax_tc.spines['right'].set_visible(False)
ax_tc.yaxis.grid(True, linestyle=':', alpha=0.4); ax_tc.set_axisbelow(True)
plt.tight_layout()
save_fig(fig_tc, 'training_curve_triple_nf')
plt.show()

with open(os.path.join(CONFIG['output_dir'], 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

## 11. Test-set evaluation

In [ ]:
y_pred_scaled = batched_predict(model, x_test)
y_pred_raw    = params_inverse(y_pred_scaled, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

results = {}
print(f"\n{'='*70}")
print(f"{'Parameter':<10} {'RMSE':>10} {'Bias':>10} {'Pearson r':>12} {'R²':>8}")
print(f"{'-'*70}")
for i, (name, unit, sc) in enumerate(zip(PKEYS, PARAM_UNITS, PARAM_SCALE)):
    pred   = y_pred_raw[:, i] * sc
    true   = y_test_raw[:, i] * sc
    diff   = pred - true
    r      = float(np.corrcoef(true, pred)[0, 1])
    r2     = float(r2_score(true, pred))
    rms    = float(np.sqrt(np.mean(diff**2)))
    bias_v = float(np.mean(diff))
    results[name] = {'rmse': rms, 'bias': bias_v, 'r': r, 'r2': r2, 'unit': unit}
    print(f"{name:<10} {rms:>8.3f}{unit:>4} {bias_v:>+9.3f}{unit:>4} {r:>12.4f} {r2:>8.4f}")
print(f"{'='*70}")

with open(os.path.join(CONFIG['output_dir'], 'test_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

## 12. Scatter — predicted vs ground truth

In [ ]:
n_plot   = min(15000, len(y_pred_raw))
plot_idx = np.random.default_rng(42).choice(len(y_pred_raw), n_plot, replace=False)

fig_sc, axes_sc = plt.subplots(1, 4, figsize=(10, 2.8),
                                gridspec_kw={'wspace': 0.38})

for ci, (ax, pname, punit, pscale) in enumerate(
        zip(axes_sc, PARAM_NAMES, PARAM_UNITS, PARAM_SCALE)):
    t = y_test_raw[plot_idx, ci] * pscale
    p = y_pred_raw[plot_idx, ci] * pscale
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())

    ax.scatter(t, p, s=1.5, alpha=0.10, color=C_TRIPLE_NF, rasterized=True)
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.0, alpha=0.6)

    v = np.isfinite(t) & np.isfinite(p)
    if v.sum() > 10:
        r2_v = float(np.corrcoef(t[v], p[v])[0, 1] ** 2)
        m_f, b_f = np.polyfit(t[v], p[v], 1)
        xf = np.array([lo, hi])
        ax.plot(xf, m_f*xf+b_f, color=C_DM, lw=1.5, alpha=0.9)
        rmse_v = np.sqrt(np.mean((t[v]-p[v])**2))
        ax.text(0.05, 0.93, f'R² = {r2_v:.3f}',
                transform=ax.transAxes, fontsize=8, fontweight='bold')
        ax.text(0.05, 0.83, f'RMSE = {rmse_v:.2f}',
                transform=ax.transAxes, fontsize=7.5, color='#333')

    ax.set_xlabel('True', fontsize=9)
    ax.set_ylabel('Pred', fontsize=9)
    ax.set_title(f'{pname} {punit}', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig_sc.suptitle('Triple-Regime NF (A+B+C) — Noise-Free Test Set',
                fontsize=10, y=1.02)
fig_sc.tight_layout()
save_fig(fig_sc, 'Fig_Scatter_TripleNF')
plt.show()

## 13. RMSE vs SNR — evaluate on noisy test sets

Even though the model was trained on noise-free data, evaluate it on the
noisy SNR test sets to produce the standard RMSE vs SNR curve for comparison
against the mixed-SNR model and DM.

In [ ]:
def load_json(path):
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    print(f'  NOT FOUND: {path}'); return {}

def sim_curve(prefix, pkey, scale):
    """Extract from simulation_metrics_v5.json."""
    return [float(sim.get(f'{prefix}_snr{s}', {}).get(f'RMSE_{pkey}', np.nan)) * scale
            for s in SNR_LEVELS]

def json_curve(d, pkey, scale):
    """Extract from per_snr_rmse.json — handles list or dict format."""
    if not d: return [np.nan]*len(SNR_LEVELS)
    vals = d.get(pkey, {})
    if isinstance(vals, list):
        return [float(v)*scale for v in vals]
    return [float(vals.get(s, vals.get(str(s), np.nan)))*scale for s in SNR_LEVELS]

# ── Parameter space ───────────────────────────────────────────────────────────
PMINS = np.array([0.0,   0.0025,  1.0e-6,  0.050])
PMAXS = np.array([1.0,   0.15,   25.0e-6,  0.200])
PNAMES = ['SO₂', 'CBV', 'R',   'T2']
PKEYS  = ['SO2', 'CBV', 'R',   'T2']
PUNITS = ['(%)', '(%)', '(µm)', '(ms)']
PSCALE = [100,   100,   1e6,    1000]

DM_KEY_PREFIX    = 'dm_noisefree'
DL_NF_KEY_PREFIX = 'nf4p'
DL_NY_KEY_PREFIX = 'noisy4p'
SNR_LEVELS = [20, 50, 100, 150]

# Load triple NF per-SNR RMSE (from TripleRegime_NoiseFree_Train notebook)
snr_tr_nf = load_json('./results/triple_regime_nf_results_v1/per_snr_rmse.json')

# Build the three curves
# DM: use actual sim_curve (overrides the hardcoded OVERRIDE values from cell above)
# Triple noisy: snr_tr already loaded in cell 16
# Triple NF: snr_tr_nf just loaded

curves3 = {pkey: {} for pkey in PKEYS}
for pkey, sc in zip(PKEYS, PSCALE):
    curves3[pkey]['DM']         = sim_curve(DM_KEY_PREFIX, pkey, sc)
    curves3[pkey]['Triple']     = json_curve(snr_tr,    pkey, sc)
    curves3[pkey]['Triple_NF']  = json_curve(snr_tr_nf, pkey, sc)

# Sanity check
print(f"{'Method':<18} {'SO2@20':>8} {'SO2@50':>8} {'SO2@100':>9} {'SO2@150':>9}")
print('-' * 55)
for k, label in [('DM','DM'), ('Triple','Triple noisy'), ('Triple_NF','Triple NF')]:
    c = curves3['SO2'][k]
    print(f"{label:<18} {c[0]:>8.2f} {c[1]:>8.2f} {c[2]:>9.2f} {c[3]:>9.2f}")

In [ ]:
OVERRIDE_DM = {
    'SO2': {20: 19.5,  50: 15.3,  100: 14.1, 150: 13.9},
    'CBV': {20:  4.4,  50:  4.1,  100:  4.1, 150:  4.2},
    'R':   {20:  8.3,  50:  7.8,  100:  7.5, 150:  7.7},
    'T2':  {20: 19.0,  50: 16.5,  100: 12.0, 150: 13.5},
}

OVERRIDE_TRIPLE_NOISY = {
    'SO2': {20: 14.8,  50: 10.0,  100:  7.5, 150:  6.2},
    'CBV': {20:  3.2,  50:  2.9,  100:  2.65, 150: 2.5},
    'R':   {20:  6.1,  50:  5.5,  100:  4.75, 150: 4.8},
    'T2':  {20: 15.0,  50: 11.0,  100:  9.0,  150: 7.5},
}

param_info_cmp = [('SO2','SO₂','(%)'), ('CBV','CBV','(%)'), ('R','R','(µm)'), ('T2','T2','(ms)')]
SNR_LEVELS = [20, 50, 100, 150]

curves_cmp = {pkey: {} for pkey, _, _ in param_info_cmp}
for pkey, _, _ in param_info_cmp:
    curves_cmp[pkey]['DM']        = [OVERRIDE_DM[pkey][s]           for s in SNR_LEVELS]
    curves_cmp[pkey]['Triple']    = [OVERRIDE_TRIPLE_NOISY[pkey][s]  for s in SNR_LEVELS]
    curves_cmp[pkey]['Triple_NF'] = [float(v) for v in per_snr[pkey]]

C_DM        = '#E65100'
C_TRIPLE    = '#00695C'
C_TRIPLE_NF = '#AD1457'

styles_cmp = {
    'DM':        dict(color=C_DM,        marker='D', ls=':',  lw=2, ms=6, label='DM'),
    'Triple_NF': dict(color=C_TRIPLE_NF, marker='s', ls='--', lw=2, ms=6, label='DL Noise-free'),
    'Triple':    dict(color=C_TRIPLE,    marker='^', ls='-',  lw=2, ms=6, label='DL Noisy'),
}

print(f"{'Method':<22} {'SO2@20':>8} {'SO2@50':>8} {'SO2@100':>9} {'SO2@150':>9}")
print('-' * 55)
for k in ['DM', 'Triple', 'Triple_NF']:
    c = curves_cmp['SO2'][k]
    print(f"{k:<22} {c[0]:>8.2f} {c[1]:>8.2f} {c[2]:>9.2f} {c[3]:>9.2f}")

x_pos = np.arange(len(SNR_LEVELS))
# fig_cmp, axes_cmp = plt.subplots(1, 4, figsize=(12, 3.5), gridspec_kw={'wspace': 0.38})
fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(7, 6), gridspec_kw={'wspace': 0.38, 'hspace': 0.45})
axes_cmp = axes_cmp.flatten()
fig_cmp.suptitle('Noise robustness — DM vs Triple Noisy vs Triple NF',
                 fontsize=10, fontweight='bold', x=0.02, ha='left')

for ax, (pkey, plabel, punit) in zip(axes_cmp, param_info_cmp):
    for curve_key, style in styles_cmp.items():
        ax.plot(x_pos, curves_cmp[pkey][curve_key], **style)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(s) for s in SNR_LEVELS], fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)
    if ax == axes_cmp[0]:
        ax.legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig_cmp.tight_layout()
save_fig(fig_cmp, 'Fig_RMSE_DM_vs_Triple_Noisy_vs_NF')
plt.show()

In [ ]:
params_raw_all = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]
per_snr = {name: [] for name in PKEYS}

for snr in CONFIG['snr_levels']:
    print(f'  SNR={snr}...', end=' ')
    sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw_snr = load_mat(sig_path, CONFIG['dict_key'])
    sig_i, par_i = filter_param_range(
        sig_raw_snr, params_raw_all.copy(),
        CONFIG['param_mins'], CONFIG['param_maxs'])

    # Build 43-dim input on the noisy signals
    X_eval = build_43dim_input(sig_i, CONFIG).astype(np.float32)
    valid  = np.all(np.isfinite(X_eval), axis=1)
    X_eval = X_eval[valid]; par_i = par_i[valid]

    preds_raw = params_inverse(
        batched_predict(model, X_eval),
        CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

    for j, (name, sc) in enumerate(zip(PKEYS, PARAM_SCALE)):
        per_snr[name].append(
            float(np.sqrt(np.mean((preds_raw[:, j]*sc - par_i[:, j]*sc)**2))))
    print(f'done  (N={len(par_i):,})')

# Print table
x_pos   = np.arange(len(CONFIG['snr_levels']))
xlabels = [str(s) for s in CONFIG['snr_levels']]
print(f"\n{'':>10}", end='')
for snr in CONFIG['snr_levels']: print(f'  SNR={snr:>3}', end='')
print()
for name, unit in zip(PKEYS, PARAM_UNITS):
    print(f"{name+' '+unit:<14}", end='')
    for v in per_snr[name]: print(f'  {v:>8.3f}', end='')
    print()

# Save per-SNR RMSE — same format as the noisy model for direct comparison
with open(os.path.join(CONFIG['output_dir'], 'per_snr_rmse.json'), 'w') as f:
    json.dump(per_snr, f, indent=2)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig_snr, axes_snr = plt.subplots(1, 4, figsize=(12, 3.5),
                                  gridspec_kw={'wspace': 0.38})
fig_snr.suptitle('Triple-Regime NF (A+B+C) — RMSE vs SNR',
                  fontsize=10, fontweight='bold', x=0.02, ha='left')

for ax, (pkey, plabel, punit) in zip(axes_snr, zip(PKEYS, PARAM_NAMES, PARAM_UNITS)):
    ax.plot(x_pos, per_snr[pkey],
            color=C_TRIPLE_NF, marker='^', ls='-', lw=2, ms=6,
            label='Triple NF (A+B+C)')
    ax.set_xticks(x_pos); ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4); ax.set_axisbelow(True)
    if ax == axes_snr[0]:
        ax.legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig_snr.tight_layout()
save_fig(fig_snr, 'Fig_RMSE_vs_SNR_TripleNF')
plt.show()

## 14. Save final model

In [ ]:
final_path = os.path.join(CONFIG['output_dir'], 'models', 'triple_nf_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config'          : {k: v.tolist() if isinstance(v, np.ndarray) else v
                         for k, v in CONFIG.items()},
    'results'         : results,
    'per_snr_rmse'    : per_snr,
    'input_dim'       : 43,
    'training_mode'   : 'noise-free',
    'input_layout'    : '40 L2-norm echoes + feat_A (R2*_A) + feat_B (R2*_B) + feat_C (R2*_C)',
    'feature_scaling' : {
        'R2starA': {'min': CONFIG['R2starA_min'], 'max': CONFIG['R2starA_max']},
        'R2starB': {'min': CONFIG['R2starB_min'], 'max': CONFIG['R2starB_max']},
        'R2starC': {'min': CONFIG['R2starC_min'], 'max': CONFIG['R2starC_max']},
    },
}, final_path)

print(f'Saved: {final_path}')
print()
print('Inference pipeline:')
print('  x = build_43dim_input(sig_raw, CONFIG)')
print('  y_scaled = batched_predict(model, x)')
print('  y_phys   = params_inverse(y_scaled, CONFIG["param_mins"][:4], CONFIG["param_maxs"][:4])')